<center>
    <h1>OCES 5303</h1>
    <h2>Assignment 1</h2>
    <hr>
    <p>Jonas Mathisrud Sterud</p>
    <p>21335836</p>
</center>

<center>
    <h5>Abstract</h5>
    <p>
    Argo is an international program collecting data from the global ocean at various depths using robotic instruments drifting with ocean currents
    <small style="margin-left: 1em">[1]</small>
    </p>
    <p>
    In this assignemnt, we will explore the dataset, and perform linear regression  to see if we can predict the sea temperature.
    </p>
    <img src="../figures/cover.jpg" width="50%">
</center>

<h1>The Data</h1>

<p>
The standard mission for Argo's robotic instruments is what's known as "park-and-profile".
</p>
<p>
It involves the robotic instrument, the "float", descending to a depth of approximatly 1,000 km (10,000 Pa) and
drifting with the ocean currents. Then, every 10 days, the float descents to 2,000 km (20,000 Pa) and starts its data collection.
It collects data on its accent back up to the surface at various depths, taking around six hours to complete.
At the surface, the float transmits the data collected over satelitte.
</p>
<p>
Most of the floats record the temperature and the salinity of the water, and some also collect information on properties describing the biology/chemistry of the water.
In this report, we'll only look at the temperature and the salinity at various depths.
</p>
<small style="margin-left: 1em">[2] [3]</small>

In [ ]:
#####################################
#       You might need to restart   #
#           the kernel after        #
#           running this cell.      #
#####################################

## Downloads

%pip install -r ./requirements.txt

In [ ]:
## Imports

import xarray as xr
import numpy as np
import pandas as pd
import datetime as dt

import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from plotly.subplots import make_subplots
from sklearn import set_config
from sklearn.preprocessing import MinMaxScaler, StandardScaler, QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, d2_absolute_error_score, r2_score
from sklearn.svm import SVC, LinearSVC

## Configuration

set_config(transform_output = 'pandas')
pio.templates.default = "plotly_dark"
seed = 42 

<h1>Parsing</h1>

<p>
We have six different variables, described in their attributes as:
</p>

<ul>
    <li><code>DEPTH</code>: Vertical Distance Below the Surface (m)</li>
    <li><code>PSAL</code>: Practical Salinity (PSU)</li>
    <li><code>TEMP</code>: Sea Temperature In-Situ ITS-90 Scale (℃)</li>
    <li><code>TIME</code>: Time of Measurement</li>
    <li><code>LATITUDE</code>: Latitude (degrees north)</li>
    <li><code>LONGITUDE</code>: Longitude (degrees east)</li>
</ul>

<p>
Here, we'll only look at data from the Atlantic Ocean, so we can remove rows with latitude and longitude outside <code>(-75, 17)</code>. In addition, we remove rows where the salinity range falls outside reasonable values.
</p>

In [ ]:
## Read and process data

ds = xr.open_zarr("../data/GLOB_HOMOGENEOUS_variables.zarr/", consolidated=False)
da_al = ds[["PSAL", "TEMP", "DEPTH"]] # (DEPTH: 302, N_PROF: 128910)
da_s  = da_al.where(((da_al.PSAL < 40.) & (da_al.PSAL > 25.)).compute(), drop=True) # Remove rows outside reasonable salinity range
da_s  = da_s.dropna('N_PROF')
da = da_s.where((da_s['LONGITUDE'] > -75.) & (da_s['LONGITUDE'] < 17), drop= True) # Atlantic ocean

df = da.to_dataframe().reset_index()
df["TIME"] = df["TIME"].to_numpy(float) # Convert to float (TODO: consider scaling)

<h1>Exploratory Data Analysis</h1>

<p>Now, let's explore the data to see if we can find any outliers, trends, etc. Let's start by plotting some of the geographical points in our dataset.</p>

In [ ]:
## Select subset of data
df_geo_subset = df.sample(200, random_state=seed)

## Scale subset of data to visualize

scalar = MinMaxScaler() # TODO: Consider other scalar
df_geo_subset["TEMP_S"] = scalar.fit_transform(df_geo_subset["TEMP"].values.reshape(-1, 1)).values
df_geo_subset["PSAL_S"] = scalar.fit_transform(df_geo_subset["PSAL"].values.reshape(-1, 1)).values

## Create figures

fig_geo_temp = px.scatter_geo(df_geo_subset, lat="LATITUDE", lon="LONGITUDE", color="TEMP_S", hover_name="TEMP")
fig_geo_psal = px.scatter_geo(df_geo_subset, lat="LATITUDE", lon="LONGITUDE", color="PSAL_S", hover_name="PSAL")

fig_geo = make_subplots(rows=1, cols=2, specs=[[{'type': 'geo'}, {'type': 'geo'}]], subplot_titles=("Temperature (scaled)", "Salinity (scaled)"))
for trace in fig_geo_temp.data: fig_geo.add_trace(trace, row=1, col=1)
for trace in fig_geo_psal.data: fig_geo.add_trace(trace, row=1, col=2)

fig_geo.update_layout(height=450, title_text="Locations of floats and their data", coloraxis_colorbar=dict(orientation="h", y=-0.2, yanchor="bottom", x=0.5, xanchor="center"))
fig_geo.show()

<p>
We see that our attempt at only selecting data from the Atlantic Ocean has failed, and we have some outliers in the Pacific Ocean.
</p>

<p>
These might interfere with our results, due to natural phenomenons such as ENSO (causing sea surface temperature differences in the Pacific Ocean). Let's remove any observations made with a longitude smaller than <code>-60.0</code> if the latitude is also smaller than <code>7.0</code>. In reality, our geographical restrictions aren't a perfect description of the Pacific Ocean, but for our data, it's good enough.
</p>

<small style="margin-left: 1em">[5]</small>

In [ ]:
## Remove geographical outliers

df = df[(df["LATITUDE"] >= 7.0) | (df["LONGITUDE"] >= -60.0)]

<p>
We can also see that we have some observations with very large values, e.g. temperature observations of around 25 ℃ (although often correlated with high salinity). We should consider removing these outliers later, but now that we have correctly selected data from the Atlantic Ocean we can train a baseline model.
</p>

<h1>Split Data</h1>

<p>
First, we'll split the data up into three different data sets: training, validation and testing. We do this to avoid any biases in our training, and minimize overfitting.
</p>

In [ ]:
## Split data (70%, 15%, 15%)

X = df.drop(columns = ["TEMP"])
y = df["TEMP"]

X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, train_size = 0.7, random_state = seed)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, train_size = 0.5, test_size = 0.5, random_state = seed)

<h1>Trends</h1>

<p>Now, let's see if we can see any obvious trends between our features and our target variables.</p>

In [ ]:
feature_names = X_train.columns
data_subset = pd.concat([X_train, y_train], axis=1).sample(int(420 * 2), random_state=seed)
fig_trends = make_subplots(rows=len(feature_names), cols=1, subplot_titles=list(map(lambda v: f"{v} vs. TEMP", feature_names)))

for (i, feat) in enumerate(feature_names):
    fig_trend = px.scatter(data_subset, x=feat, y="TEMP", color="TEMP", trendline="ols")
    fig_trend.update_traces(line={"color": "orange", "width": 2, "dash": "dot"})
    for trace in fig_trend.data: fig_trends.add_trace(trace, row=(i + 1), col=1)

fig_trends.update_layout(height=1500, coloraxis_showscale=False, coloraxis=dict(colorscale="Plasma"))
fig_trends.show()

<p>We see that <code>PSAL</code> is the feature with the strongest correlation to <code>TEMP</code>.</p>

<h1>Baseline Model</h1>

<p>
Now, we'll train a baseline model so we have something to compare our models against.
The baseline model will simply calculate the mean of our target variable.
</p>

In [ ]:
## Train baseline model

r_dummy = DummyRegressor()
r_dummy = r_dummy.fit(X_train, y_train)

## Visualize

metrics_dummy = dict({
    "Mean Absolute Error": mean_absolute_error(y_val, r_dummy.predict(X_val)),
    "Mean Squared Error": np.sqrt(mean_squared_error(y_val, r_dummy.predict(X_val))),
})

for k, v in metrics_dummy.items():
    print(f"{k}: {v:.5f}")

Mean Absolute Error: 4.30499
Mean Squared Error: 5.47931


<h1>Linear Regression</h1>

<p>Let's see if linear regression performs any better.</p>

In [ ]:
## Train linear regression model

r_linear = LinearRegression()
r_linear = r_linear.fit(X_train, y_train)

## Visualize

metrics_linear = dict({
    "Mean Absolute Error": mean_absolute_error(y_val, r_linear.predict(X_val)),
    "Mean Squared Error": np.sqrt(mean_squared_error(y_val, r_linear.predict(X_val))),
})

for k, v in metrics_linear.items():
    print(f"{k}: {v:.5f}")

Mean Absolute Error: 4.30503
Mean Squared Error: 5.47904


<p>Perhaps a bit suprisingly, the linear regression struggles to find any relationship between the features and the target variable, and our performance results are almost identical to our <code>DummyRegressor</code>.</p>

<p>We should perhaps consider feature selection, clustering, scaling, and other machine learning methods to improve our model.</p>

<p>But first, let's try a few other models.</p>

<h1>Ridge Regression</h1>

<p>This regression model also uses least squares, but includes L2 regularization.</p>

In [ ]:
## Train logistic regression model

r_ridge = Ridge(random_state=seed)
r_ridge = r_ridge.fit(X_train, y_train)

## Visualize

metrics_ridge = dict({
    "Mean Absolute Error": mean_absolute_error(y_val, r_ridge.predict(X_val)),
    "Mean Squared Error": np.sqrt(mean_squared_error(y_val, r_ridge.predict(X_val))),
})

for k, v in metrics_ridge.items():
    print(f"{k}: {v:.5f}")

Mean Absolute Error: 1.62251
Mean Squared Error: 2.30666


/usr/local/conda/envs/OCES5303/lib/python3.14/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.633875988394458e-35.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


<h1>Random Forest Regression</h1>

In [ ]:
## Train random forest regression model

r_randomforest = RandomForestRegressor(n_estimators=10, max_depth=5, n_jobs=-1, random_state=seed)
r_randomforest = r_randomforest.fit(X_train, y_train)

## Visualize

metrics_randomforest = dict({
    "Mean Absolute Error": mean_absolute_error(y_val, r_randomforest.predict(X_val)),
    "Mean Squared Error": np.sqrt(mean_squared_error(y_val, r_randomforest.predict(X_val))),
})

for k, v in metrics_randomforest.items():
    print(f"{k}: {v:.5f}")

Mean Absolute Error: 0.87750
Mean Squared Error: 1.34960


<h1>Feature Selection & Scaling Features</h1>

<p>One reason that we might be seeing better results using random forest regression, is that it's scale-invariant. Unlike ridge regression, where scaling of the features yields different results, random forest's node based are not affected by scaled features, since the decision nodes will just scale with the features.</p>

<p>In addition, we have so far trained our regression models with all the available features. But, features like <code>LATITUDE</code> and <code>LONGITUDE</code>, are high-cardinality features. They are variables used to describe the location of where a measurement was made, but a machine learning model might struggle to generalize this data to new unseen data, since it's specific to each data point.</p>

<p>Lastly, the <code>N_PROF</code> feature is an identifier for each data point, which will also make it hard for machine learning models to generalize for unseen data.</p>

<p>We'll create a pipeline that removes these features, scales the other features appropriately, and tests different machine learning models.</p>

In [ ]:
df

,N_PROF,DEPTH,PSAL,TEMP,TIME,LATITUDE,LONGITUDE
19026,63,0.0,36.543999,29.337986,1.217684e+18,27.378000,-74.089996
19027,63,-5.0,36.543999,29.337986,1.217684e+18,27.378000,-74.089996
19028,63,-10.0,36.544018,29.336027,1.217684e+18,27.378000,-74.089996
19029,63,-15.0,36.546902,29.334730,1.217684e+18,27.378000,-74.089996
19030,63,-20.0,36.615002,29.160534,1.217684e+18,27.378000,-74.089996
...,...,...,...,...,...,...,...
9931567,32885,-1485.0,38.740974,13.753141,1.371849e+18,37.502998,16.094000
9931568,32885,-1490.0,38.740871,13.753646,1.371849e+18,37.502998,16.094000
9931569,32885,-1495.0,38.740772,13.754150,1.371849e+18,37.502998,16.094000
9931570,32885,-1500.0,38.740669,13.754655,1.371849e+18,37.502998,16.094000


In [ ]:
pipeline = Pipeline([
    #("remove_features", ColumnTransformer([("remove_features", "drop", ["N_PROF", "LATITUDE", "LONGITUDE"])], remainder="passthrough", verbose_feature_names_out = False)),
    ("scale_standard", ColumnTransformer([("scale_standard", StandardScaler(), [])], remainder="passthrough", verbose_feature_names_out=False)),
    ("scale_minmax", ColumnTransformer([("scale_minmax", MinMaxScaler(), [])], remainder="passthrough", verbose_feature_names_out=False)),
    ("scale_quantile", ColumnTransformer([("scale_minmax", QuantileTransformer(random_state=seed), [])], remainder="passthrough", verbose_feature_names_out=False)),
    ("regression", Ridge(random_state = seed))
])

pipeline.fit(X_train, y_train)

metrics_pipeline = dict({
    "Mean Absolute Error": mean_absolute_error(y_val, pipeline.predict(X_val)),
    "Mean Squared Error": np.sqrt(mean_squared_error(y_val, pipeline.predict(X_val))),
})

for k, v in metrics_pipeline.items():
    print(f"{k}: {v:.5f}")

# 1.62251
# 2.30666

Mean Absolute Error: 1.62251
Mean Squared Error: 2.30666


/usr/local/conda/envs/OCES5303/lib/python3.14/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 2.633875988394458e-35.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


In [ ]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scale_standard', ...), ('scale_minmax', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale_standard', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differe

<h1>Clustering</h1>

In [ ]:
df.sample(10)

,N_PROF,DEPTH,PSAL,TEMP,TIME,LATITUDE,LONGITUDE
7438981,24632,-585.0,34.597240,1.912310,1.271941e+18,-52.077000,-26.632999
5653408,18719,-1350.0,34.994617,4.679208,1.180446e+18,38.622002,-55.695000
3622423,11994,-1175.0,35.499565,10.105613,1.379967e+18,40.033001,-15.950000
945782,3131,-1100.0,35.142700,6.229314,1.356034e+18,30.004000,-47.126999
782735,2591,-1265.0,34.341984,3.026073,1.348240e+18,-45.099998,-49.669998
3036229,10053,-1115.0,34.511955,3.436573,1.222082e+18,-24.445000,-21.611000
9892625,32757,-55.0,35.527000,19.357843,1.215836e+18,-39.819000,15.609000
7938291,26285,-1105.0,34.913364,4.009823,1.327640e+18,57.611000,-23.648001
1477523,4892,-695.0,35.535442,11.206804,1.363343e+18,25.125999,-38.339001
6937135,22970,-975.0,34.366669,2.886524,1.184333e+18,-44.034000,-32.192001


In [ ]:
px.scatter(df.sample(1500), x="PSAL", y="TEMP")

In [ ]:
c_supportvector = SVC()
c_supportvector.fit(X_train, y_train)

pred = c_supportvector.predict(X_val)

ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.

<h1>References</h1>

<p>[1]    Argo. <a href="https://argo.ucsd.edu" target="_blank">https://argo.ucsd.edu</a>. 2026.</p>
<p>[2]    Argo Program Office. <i>About Argo</i>. <a href="https://argo.ucsd.edu/about" target="_blank">https://argo.ucsd.edu/about</a> [Online; accessed 26-Februrary-2026]. 2026.</p>
<p>[3]    Wong, Annie P. S., et. al. <i>Argo Data 1999-2019: Two Million Temperature-Salinity
Profiles and Subsurface Velocity Observations From a Global Array of Profiling Floats</i>. <a href="https://www.frontiersin.org/journals/marine-science/articles/10.3389/fmars.2020.00700" target="_blank">https://www.frontiersin.org/journals/marine-science/articles/10.3389/fmars.2020.00700</a> [Online; accessed 26-Februrary-2026]. 2026.</p>
<p>[4]    Argo (2000). <i>Argo float data and metadata from Global Data Assembly Centre (Argo GDAC)</i>. SEANOE. <a href="http://doi.org/10.17882/42182" target="_blank">http://doi.org/10.17882/42182</a></p>
<p>[5]    Wikipedia contributors (2026). <i>El Niño–Southern Oscillation</i>. Wikipedia, The Free Encyclopedia. <a href="https://en.wikipedia.org/w/index.php?title=El_Ni%C3%B1o%E2%80%93Southern_Oscillation&oldid=1341681100" target="_blank">https://en.wikipedia.org/w/index.php?title=El_Ni%C3%B1o%E2%80%93Southern_Oscillation&oldid=1341681100</a> [Online; accessed 5-March-2026</p>